In [1]:
import re
from langchain.chat_models import ChatOpenAI
from langchain_community.chat_models import ChatHuggingFace
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.llms.base import BaseLLM

In [3]:
from dotenv import load_dotenv
load_dotenv()
def build_llm(model_name: str, temperature: float = 0.7) -> BaseLLM:
    """
    根據 model_name 回傳對應的 LLM 物件

    支援的 model_name: "gpt-3.5", "gpt-4o", "llama3", "qwen"
    """

    model_name = model_name.lower()

    if model_name == "gpt-3.5":
        llm: BaseLLM = ChatOpenAI(model="gpt-3.5-turbo", temperature=temperature)

    elif model_name == "gpt-4o":
        llm: BaseLLM = ChatOpenAI(model="gpt-4o", temperature=temperature)

    elif model_name == "gemini":
        llm: BaseLLM = ChatGoogleGenerativeAI(model="gemini-2.0-flash-001", temperature=temperature)

    # elif model_name == "llama3":
    #     llm: BaseLLM = ChatHuggingFace(
    #         repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    #         task="text-generation",
    #         model_kwargs={"temperature": temperature, "max_new_tokens": 4096}
    #     )

    # elif model_name.lower() == "qwen":
    #     llm: BaseLLM = ChatHuggingFace(
    #         repo_id="huihui-ai/Qwen2.5-72B-Instruct-abliterated",
    #         task="text-generation",
    #         model_kwargs={"temperature": temperature, "max_new_tokens": 4096}
    #     )

    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

    return llm

In [4]:
llm = build_llm("gemini")

In [8]:
! huggingface-cli download joe32140/ModernBERT-large-msmarco

Fetching 19 files:   0%|                                 | 0/19 [00:00<?, ?it/s]Downloading '.gitattributes' to '/Users/jerryshih/.cache/huggingface/hub/models--joe32140--ModernBERT-large-msmarco/blobs/a6344aac8c09253b3b630fb776ae94478aa0275b.incomplete'

.gitattributes: 100%|██████████████████████| 1.52k/1.52k [00:00<00:00, 10.5MB/s]
Download complete. Moving file to /Users/jerryshih/.cache/huggingface/hub/models--joe32140--ModernBERT-large-msmarco/blobs/a6344aac8c09253b3b630fb776ae94478aa0275b
Fetching 19 files:   5%|█▎                       | 1/19 [00:00<00:08,  2.10it/s]Downloading 'onnx/model_fp16.onnx' to '/Users/jerryshih/.cache/huggingface/hub/models--joe32140--ModernBERT-large-msmarco/blobs/5c0eefb973837d85de94c1fedb1dc040ee4a715a8649d2d5ff539b8111d184b4.incomplete'

model_q4f16.onnx:   0%|                              | 0.00/297M [00:00<?, ?B/s]

model_int8.onnx:   0%|                               | 0.00/397M [00:00<?, ?B/s]Downloading 'onnx/model_uint8.onnx' to '/Users/jerr

In [12]:

from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer("/Users/jerryshih/.cache/huggingface/hub/models--joe32140--ModernBERT-large-msmarco/snapshots/a22186df5a74bc8b73f3cdd1755df2e01e37c565")

In [7]:
import json
# import 
birth_info = {}
user_input = "1996 11 06 16"
prompt = f"""
    <rule>
    你是有用的資料整理人員, 請從以下使用者輸入中，抽取出生的「年份、月份、日期、時辰」，return JSON 格式,
    如果有任何欄位缺失，請用 null 表示。
    </rule>

    <hour_mapping>
    此外, hour 使用者可能是輸入地支: 子丑寅卯.., 
    hour 的對照表如下:
        子:  hour 設為 '0'
        丑:  hour 設為 '2'
        寅:  hour 設為 '4'
        卯:  hour 設為 '6'
        辰:  hour 設為 '8'
        巳:  hour 設為 '10'
        午:  hour 設為 '12'
        未:  hour 設為 '14'
        申:  hour 設為 '16'
        酉:  hour 設為 '18'
        戌:  hour 設為 '20'
        亥:  hour 設為 '22'
    </hour_mapping>
    
    <input>
    {user_input}
    </input>

    <output example>
    1. {{"year":"1999","month":"2","day":"23","hour":"2"}}
    2. {{"year":"1994","month":"11","day":"21","hour":"16"}}
    3. {{"year":"1998","month":"8","day":"27","hour":"10"}}
    4. {{"year":"2000","month":"3","day":"22","hour":"12"}}
    </output example>

    <output format>
    輸出除了 json 檔, 不包含任何 、\n(換行)或斜線以及空格等特殊字元、
    可以直接將 return 字串給 python load, eg: json.load(output)
    </output format>

"""

response = llm.invoke(prompt)
print("response: ", response)
# response = response.replace(" ", "").replace("\n", "")
# print("[get_birth_info] 解析 input 結果", response)
try:
    json_str = response.content
    match = re.search(r'\{.*?\}', json_str)
    if match:
        json_str = match.group()
    extracted = json.loads(json_str)
    # print("[get_birth_info] extracted", extracted)
    for key in ["year", "month", "day", "hour"]:
        if extracted.get(key):
            birth_info[key] = int(extracted[key])
except Exception:
    pass  # 若 LLM 回傳不是 JSON，就略過這次抽取

response:  content='```json\n{"year":"1996","month":"11","day":"06","hour":"16"}\n```' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-001', 'safety_ratings': []} id='run-deada1b4-f52c-4935-b65d-e34b901a646e-0' usage_metadata={'input_tokens': 451, 'output_tokens': 28, 'total_tokens': 479, 'input_token_details': {'cache_read': 0}}


In [10]:
review_list = ["a", "b"]
''.join(review_list)

'ab'

In [8]:
birth_info

{'year': 1996, 'month': 11, 'day': 6, 'hour': 16}